## Notebook to show how to read a 3DSG from the 3DSSG dataset, print information and plot 3D


In [12]:
import json
import logging
import os
import sys

os.environ["XDG_SESSION_TYPE"] = "x11"
os.environ["OPEN3D_DISABLE_WEB_VISUALIZER"] = "true"     

import numpy as np
import open3d as o3d

sys.path.append('../src')
from SceneGraph3D import SceneGraph3D

%load_ext autoreload
%autoreload 2
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Constants definition


In [13]:
SSG_REPO = "../3DSSG"  # Path to the 3DSSG repository
RSCAN_REPO = "../3RScan"  # Path to the 3RScan repository
REL_FILE = os.path.join(SSG_REPO, "relationships.json")
OBJ_FILE = os.path.join(SSG_REPO, "objects.json")
SEMSEG_FILE = "semseg.v2.json"
PCD_FILE = "labels.instances.annotated.v2.ply"
MESH_FILE = "mesh.refined.v2.obj"
# change this to explore other scans (you can also cycle through them)
SCAN_ID = 882  # Example of one scan ID in 3RScan
SCAN_UUID = "02b33e03-be2b-2d54-9129-5d28efdd68fa" # "0cac75a7-8d6f-2d13-8fdc-083ff44d10fb" # "e2847ff7-a506-2b60-869c-2780e8694ae0" # "f4f315fe-8408-2255-974c-e485355f9f9d" # "fcf66dae-622d-291c-862d-dc03f6f1d562" # "0ad2d3a3-79e2-2212-9a51-9094be707ec2"

## Load the 3DSSG dataset


In [4]:
scans = json.load(open(REL_FILE))["scans"]
objects = json.load(open(OBJ_FILE))["scans"]
objects = {obj["scan"]: obj for obj in objects}
for scan in scans:
    id_scan = scan["scan"]
    scan["objects"] = objects[id_scan]["objects"]
max_objects_num = 0
max_rel_num = 0
min_objects_num = 20
min_rel_num = 200
for scan in scans:
    if len(scan["relationships"]) == 0:
        print(scan)
    max_objects_num = max(max_objects_num, len(scan["objects"]))
    max_rel_num = max(max_rel_num, len(scan["relationships"]))
    min_objects_num = min(min_objects_num, len(scan["objects"]))
    min_rel_num = min(min_rel_num, len(scan["relationships"]))
print(max_objects_num, max_rel_num)  # 147 5167
print(min_objects_num, min_rel_num)  # 2 1

147 5167
2 1


In [5]:
def index_by_uuid(scans, uuid):
    for scan in scans:
        if scan.get("scan", None) == uuid:
            return scan
    raise KeyError(f"Unknown scan uuid {uuid}")

In [6]:
print(scans[SCAN_ID]["objects"][0].keys())
print(scans[SCAN_ID]["relationships"])

dict_keys(['ply_color', 'nyu40', 'eigen13', 'label', 'rio27', 'affordances', 'id', 'global_id', 'attributes'])
[[2, 1, 15, 'standing on'], [5, 1, 14, 'attached to'], [6, 1, 14, 'attached to'], [7, 1, 15, 'standing on'], [8, 2, 15, 'standing on'], [9, 2, 1, 'supported by'], [10, 1, 15, 'standing on'], [11, 10, 16, 'lying on'], [2, 7, 6, 'close by'], [2, 10, 3, 'right'], [2, 10, 6, 'close by'], [7, 2, 6, 'close by'], [7, 10, 3, 'right'], [7, 10, 6, 'close by'], [8, 9, 5, 'behind'], [8, 9, 6, 'close by'], [8, 9, 2, 'left'], [9, 8, 4, 'front'], [9, 8, 3, 'right'], [9, 8, 6, 'close by'], [10, 2, 6, 'close by'], [10, 2, 2, 'left'], [10, 7, 6, 'close by'], [10, 7, 2, 'left'], [5, 6, 32, 'same object type'], [10, 7, 32, 'same object type'], [6, 5, 32, 'same object type'], [7, 10, 32, 'same object type']]


## Instantiating the Graph


In [7]:
g = SceneGraph3D.from_dict(index_by_uuid(scans, SCAN_UUID))
scan_id = g.scan_id
file_3d_path = os.path.join(RSCAN_REPO, scan_id, SEMSEG_FILE)
file_pointcloud = os.path.join(RSCAN_REPO, scan_id, PCD_FILE)
file_mesh = os.path.join(RSCAN_REPO, scan_id, MESH_FILE)
g.read_3d_scene(file_3d_path, file_pointcloud, file_mesh)


### Printing all the nodes id of the graph


In [24]:
objects = g.nodes(data=True)
print("List of objects:\n", objects)
print("Num objects:", len(objects))

List of objects:
 [(1, {'ply_color': '#aec7e8', 'nyu40': '2', 'eigen13': '5', 'label': 'floor', 'rio27': '2', 'affordances': ['placing items on', 'walking on'], 'id': '1', 'global_id': '188', 'attributes': {'texture': ['tiled'], 'material': ['ceramic'], 'shape': ['flat'], 'lexical': ['inside', 'lower', 'horizontal'], 'color': ['brown']}}), (2, {'ply_color': '#1f77b4', 'nyu40': '1', 'eigen13': '12', 'label': 'wall', 'rio27': '1', 'affordances': ['leaning against', 'placing items on'], 'id': '2', 'global_id': '503', 'attributes': {'shape': ['sloping'], 'lexical': ['architectural'], 'color': ['white']}}), (3, {'ply_color': '#ffbb78', 'nyu40': '9', 'eigen13': '13', 'state_affordances': ['opening (closed)'], 'label': 'window', 'rio27': '9', 'affordances': ['looking outside', 'throwing out of'], 'id': '3', 'global_id': '520', 'attributes': {'material': ['glass'], 'state': ['closed'], 'shape': ['semicircular'], 'color': ['white'], 'size': ['tall']}}), (4, {'ply_color': '#ff7f0e', 'nyu40': '9'

In [21]:
position_5 = g.get_node_centroid(5)
print(position_5)

[-2.397921004774094, 2.163888805814448, -0.17701015767351214]


### Printing all the edges of the graph


In [22]:
edges = g.edges(data=True)
print("List of edges:\n", edges)
print("Num edges:", len(edges))

List of edges:
 [(1, 54, {'id': 27, 'name': 'same color'}), (1, 12, {'id': 30, 'name': 'same shape'}), (1, 22, {'id': 30, 'name': 'same shape'}), (1, 33, {'id': 27, 'name': 'same color'}), (1, 6, {'id': 30, 'name': 'same shape'}), (1, 40, {'id': 30, 'name': 'same shape'}), (1, 28, {'id': 30, 'name': 'same shape'}), (1, 17, {'id': 30, 'name': 'same shape'}), (1, 26, {'id': 30, 'name': 'same shape'}), (1, 19, {'id': 30, 'name': 'same shape'}), (1, 16, {'id': 30, 'name': 'same shape'}), (1, 7, {'id': 30, 'name': 'same shape'}), (1, 21, {'id': 30, 'name': 'same shape'}), (1, 20, {'id': 30, 'name': 'same shape'}), (1, 8, {'id': 30, 'name': 'same shape'}), (1, 5, {'id': 30, 'name': 'same shape'}), (1, 39, {'id': 30, 'name': 'same shape'}), (1, 23, {'id': 30, 'name': 'same shape'}), (2, 1, {'id': 14, 'name': 'attached to'}), (2, 14, {'id': 30, 'name': 'same shape'}), (2, 5, {'id': 27, 'name': 'same color'}), (2, 22, {'id': 32, 'name': 'same object type'}), (2, 13, {'id': 27, 'name': 'same col

### Printing the names of the entities of the graph


In [23]:
for obj in objects:
    print("e -> ", obj[1]["label"])

e ->  floor
e ->  wall
e ->  window
e ->  window
e ->  wall
e ->  wall
e ->  wall
e ->  wall
e ->  wall
e ->  mirror
e ->  wall
e ->  door
e ->  wall
e ->  wall
e ->  wall
e ->  wall
e ->  ceiling
e ->  sofa
e ->  wall
e ->  wall
e ->  wall
e ->  wall
e ->  door
e ->  stairs
e ->  tv stand
e ->  tv
e ->  bench
e ->  wall
e ->  shelf
e ->  plant
e ->  table
e ->  chair
e ->  chair
e ->  wall
e ->  table
e ->  clothes
e ->  ceiling
e ->  door
e ->  box
e ->  item
e ->  item
e ->  item
e ->  shelf
e ->  pillow
e ->  pillow
e ->  pillow
e ->  bag
e ->  pillow
e ->  table
e ->  doorframe
e ->  doorframe
e ->  stand
e ->  plant
e ->  plant
e ->  rack
e ->  vacuum cleaner
e ->  pillow
e ->  pillow
e ->  sidecouch


### Printing the relations of the edge


In [25]:
for edj in edges:
    e1 = g[edj[0]]["label"]
    e2 = g[edj[1]]["label"]
    edj_name = edj[2]["name"]
    print(e1, " -> ", edj[2]["name"], " -> ", e2)
print("Num objects:", len(objects))

floor  ->  same color  ->  table
floor  ->  same shape  ->  door
floor  ->  same shape  ->  wall
floor  ->  same color  ->  table
floor  ->  same shape  ->  wall
floor  ->  same shape  ->  door
floor  ->  same shape  ->  wall
floor  ->  same shape  ->  ceiling
floor  ->  same shape  ->  tv
floor  ->  same shape  ->  wall
floor  ->  same shape  ->  wall
floor  ->  same shape  ->  wall
floor  ->  same shape  ->  wall
floor  ->  same shape  ->  wall
floor  ->  same shape  ->  wall
floor  ->  same shape  ->  wall
floor  ->  same shape  ->  ceiling
floor  ->  same shape  ->  door
wall  ->  attached to  ->  floor
wall  ->  same shape  ->  wall
wall  ->  same color  ->  wall
wall  ->  same object type  ->  wall
wall  ->  same color  ->  wall
wall  ->  brighter than  ->  tv
wall  ->  same object type  ->  wall
wall  ->  brighter than  ->  box
wall  ->  same color  ->  door
wall  ->  same object type  ->  wall
wall  ->  same color  ->  wall
wall  ->  same object type  ->  wall
wall  ->  same ob

In [13]:
# I want to count the different types of edges
edge_types = {}
for edj in edges:
    e1 = g[edj[0]]["label"]
    e2 = g[edj[1]]["label"]
    edj_name = edj[2]["name"]
    if edj_name not in edge_types:
        edge_types[edj_name] = 1
    else:
        edge_types[edj_name] += 1
print("Edge types and counts:")
for k, v in edge_types.items():
    print(f"{k}: {v}")


Edge types and counts:
same shape: 33
same color: 15
same object type: 4
standing on: 5
left: 8
behind: 3
hanging on: 1
supported by: 1
right: 4
close by: 3
attached to: 2


#### Plot the graph in 2D

You may need to install posix graphviz https://stackoverflow.com/questions/35064304/runtimeerror-make-sure-the-graphviz-executables-are-on-your-systems-path-aft


In [14]:
SceneGraph3D.render(g, "../outputs/example_plot")  # renders to example_plot.pdf

#### Saving the graph as JSON (open and explore the JSON)


In [15]:
SceneGraph3D.to_json(g, "../outputs/example_output.json")

## Visualize 3D Object semantic segmented


In [16]:
vertex = g.pointcloud["vertex"]
xyz = np.vstack((vertex["x"], vertex["y"], vertex["z"])).T
colors = np.vstack((vertex["red"], vertex["green"], vertex["blue"])).T
obj_pcd = o3d.geometry.PointCloud()
obj_pcd.points = o3d.utility.Vector3dVector(xyz)
obj_pcd.colors = o3d.utility.Vector3dVector(colors / 255.0)
o3d.visualization.draw_plotly([obj_pcd])


In [17]:
obj7 = g.get_object_pointcloud(7)  # get point cloud of object with node ID 7
o3d.visualization.draw_plotly([obj7])
# you could extract from this point cloud feautures with for example PointNet

### Visualize Mesh


In [11]:
o3d.visualization.draw([{"geometry": g.mesh, "name": "3RScan Mesh"}], show_ui=True)

FEngine (64 bits) created at 0x3811e980 (threading is enabled)
FEngine resolved backend: OpenGL


In [17]:
o3d.visualization.draw_geometries([g.mesh])

In [12]:
o3d.visualization.draw_plotly([g.mesh])

In [19]:
mesh = g.mesh

colors = np.asarray(mesh.vertex_colors)

# convert to float and scale
colors = colors.astype(float) / 255.0

mesh.vertex_colors = o3d.utility.Vector3dVector(colors)

o3d.visualization.draw_plotly([mesh])